In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import torch.cuda.amp as amp  # For mixed precision training
from tqdm import tqdm  # For progress bar

class VanillaCNN(nn.Module):
    def __init__(self, num_classes):
        super(VanillaCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 64 * 64, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def train_model(train_dir, batch_size=32, epochs=10, initial_lr=0.001, val_split=0.2, device="cuda" if torch.cuda.is_available() else "cpu"):
    # Define transforms to apply to the images (convert to tensor and normalize)
    transform = transforms.Compose([
        transforms.Resize((128, 128)),  # Resize if needed
        transforms.ToTensor(),  # Convert to tensor
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize with ImageNet stats or your own
    ])

    # Load dataset with the transformation
    dataset = datasets.ImageFolder(train_dir, transform=transform)

    train_size = int((1 - val_split) * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    model = VanillaCNN(num_classes=len(dataset.classes)).to(device)
    criterion = nn.CrossEntropyLoss()
    my_lr = initial_lr
    optimizer = optim.SGD(model.parameters(), lr=my_lr)
    scaler = amp.GradScaler()  # Mixed precision training scaler

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        if epoch % 5 == 0:
            my_lr /= 1.5
            optimizer = optim.SGD(model.parameters(), lr=my_lr)

        model.train()
        running_loss, running_error, num_batches = 0, 0, 0

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}")
        for minibatch_data, minibatch_label in progress_bar:
            optimizer.zero_grad()

            minibatch_data = minibatch_data.to(device, non_blocking=True)
            minibatch_label = minibatch_label.to(device, non_blocking=True)

            with amp.autocast():  # Mixed precision
                outputs = model(minibatch_data)
                loss = criterion(outputs, minibatch_label)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.detach().item()
            _, predicted = torch.max(outputs.detach(), 1)
            running_error += (predicted != minibatch_label).sum().item()
            num_batches += 1

            progress_bar.set_postfix(loss=running_loss / num_batches, error=running_error / len(train_dataset) * 100)

        total_loss = running_loss / num_batches
        total_error = running_error / len(train_dataset)
        elapsed_time = (time.time() - start_time) / 60

        print(f"Epoch={epoch}, Time={elapsed_time:.2f} min, LR={my_lr:.6f}, Loss={total_loss:.4f}, Error={total_error * 100:.2f}%")

        # Evaluate on validation data
        evaluate_model(model, val_loader, device, mode='Validation')

    return model

def evaluate_model(model, data_loader, device="cuda" if torch.cuda.is_available() else "cpu", mode='Test'):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        progress_bar = tqdm(data_loader, desc=f"Evaluating {mode}")
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    accuracy = 100 * correct / total
    print(f"{mode} Accuracy: {accuracy:.2f}%")
    return accuracy

In [ ]:
# train and evaluate the model
device= torch.device("cuda")
train_dir = '/kaggle/input/foodimages/datasets/train'
test_dir = '/kaggle/input/foodimages/datasets/test'
batch_size = 100
epochs = 10
initial_lr = 0.03

# Train the model
VanillaCnnModel = train_model(train_dir, batch_size, epochs, initial_lr, device=device)

<ipython-input-30-566d12389557>:50: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()  # Mixed precision training scaler
Epoch 1:   0%|          | 0/247 [00:00<?, ?it/s]<ipython-input-30-566d12389557>:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():  # Mixed precision
Epoch 1: 100%|██████████| 247/247 [00:31<00:00,  7.81it/s, error=97.2, loss=4.31]


Epoch=1, Time=0.53 min, LR=0.030000, Loss=4.3108, Error=97.15%


Evaluating Validation: 100%|██████████| 62/62 [00:07<00:00,  7.81it/s]


Validation Accuracy: 5.32%


Epoch 2: 100%|██████████| 247/247 [00:31<00:00,  7.95it/s, error=92.6, loss=3.94]


...

Validation Accuracy: 15.63%


Epoch 10: 100%|██████████| 247/247 [00:31<00:00,  7.89it/s, error=59, loss=2.26]  


Epoch=10, Time=6.43 min, LR=0.013333, Loss=2.2626, Error=59.04%


Evaluating Validation: 100%|██████████| 62/62 [00:07<00:00,  7.80it/s]

Validation Accuracy: 16.49%


In [ ]:
# Load test data with transformations
test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Evaluate the model
evaluate_model(VanillaCnnModel, test_loader, device)

Evaluating Test: 100%|██████████| 78/78 [00:31<00:00,  2.47it/s]

Test Accuracy: 16.47%


16.47196261682243

In [ ]:
# Save the model
def save_model(model, filepath):
    torch.save(model.state_dict(), filepath)
    print(f"Model saved to {filepath}")

save_model(VanillaCnnModel, 'vanilla_cnn_model.pth')  # Save only the state_dict

Model saved to vanilla_cnn_model.pth


Lenet5

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5, stride=1, padding=2)
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1)

        self.fc1 = nn.Linear(16 * 60 * 60, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

def train_model(train_dir, batch_size=32, epochs=10, initial_lr=0.001, val_split=0.2, device="cuda" if torch.cuda.is_available() else "cpu"):
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    dataset = datasets.ImageFolder(train_dir, transform=transform)

    train_size = int((1 - val_split) * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    model = LeNet5(num_classes=len(dataset.classes)).to(device)
    criterion = nn.CrossEntropyLoss()
    my_lr = initial_lr
    optimizer = optim.SGD(model.parameters(), lr=my_lr)
    scaler = amp.GradScaler()

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        if epoch % 5 == 0:
            my_lr /= 1.5
            optimizer = optim.SGD(model.parameters(), lr=my_lr)

        model.train()
        running_loss, running_error, num_batches = 0, 0, 0

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}")
        for minibatch_data, minibatch_label in progress_bar:
            optimizer.zero_grad()

            minibatch_data = minibatch_data.to(device, non_blocking=True)
            minibatch_label = minibatch_label.to(device, non_blocking=True)

            with amp.autocast():
                outputs = model(minibatch_data)
                loss = criterion(outputs, minibatch_label)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.detach().item()
            _, predicted = torch.max(outputs.detach(), 1)
            running_error += (predicted != minibatch_label).sum().item()
            num_batches += 1

            progress_bar.set_postfix(loss=running_loss / num_batches, error=running_error / len(train_dataset) * 100)

        total_loss = running_loss / num_batches
        total_error = running_error / len(train_dataset)
        elapsed_time = (time.time() - start_time) / 60

        print(f"Epoch={epoch}, Time={elapsed_time:.2f} min, LR={my_lr:.6f}, Loss={total_loss:.4f}, Error={total_error * 100:.2f}%")

        evaluate_model(model, val_loader, device, mode='Validation')

    return model

In [ ]:
batch_size = 100
epochs = 10
initial_lr = 0.03
Lenet5Model = train_model(train_dir, batch_size, epochs, initial_lr, device=device)

<ipython-input-26-0b263ab9df56>:49: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()
Epoch 1:   0%|          | 0/247 [00:00<?, ?it/s]<ipython-input-26-0b263ab9df56>:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
Epoch 1: 100%|██████████| 247/247 [00:31<00:00,  7.76it/s, error=97.8, loss=4.32]


Epoch=1, Time=0.53 min, LR=0.030000, Loss=4.3204, Error=97.78%


Evaluating Validation: 100%|██████████| 62/62 [00:08<00:00,  7.41it/s]


Validation Accuracy: 2.68%


Epoch 2: 100%|██████████| 247/247 [00:31<00:00,  7.90it/s, error=94.2, loss=4.05]


...

Validation Accuracy: 14.44%


Epoch 10: 100%|██████████| 247/247 [00:31<00:00,  7.76it/s, error=71.4, loss=2.82]


Epoch=10, Time=6.45 min, LR=0.013333, Loss=2.8239, Error=71.37%


Evaluating Validation: 100%|██████████| 62/62 [00:08<00:00,  7.40it/s]

Validation Accuracy: 16.18%


In [ ]:
# Evaluate the model
evaluate_model(Lenet5Model, test_loader, device)

Evaluating Test: 100%|██████████| 78/78 [00:32<00:00,  2.40it/s]

Test Accuracy: 15.90%


15.900830737279335

In [ ]:
save_model(Lenet5Model, 'lenet5_model.pth')  # Save only the state_dict

Model saved to lenet5_model.pth


Vgg16

In [ ]:
import torchvision.models as models
# Load the VGG16 model
class VGG16(nn.Module):
    def __init__(self, num_classes):
        super(VGG16, self).__init__()
        self.model = models.vgg16(pretrained=True)
        self.model.classifier[6] = nn.Linear(4096, num_classes)  # Modify the last layer for our dataset

    def forward(self, x):
        return self.model(x)

# Function to train the model
def train_model(train_dir, batch_size=32, epochs=10, initial_lr=0.001, val_split=0.2, device="cuda" if torch.cuda.is_available() else "cpu"):
    """
    Train VGG16 on image dataset
    - Splits training data into train/validation sets
    - Uses dynamic learning rate adjustment
    - Implements mixed precision training for efficiency
    """

    # Define preprocessing transformations
    transform = transforms.Compose([
        transforms.Resize((224, 224)),  # Resize for VGG16
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Load dataset
    dataset = datasets.ImageFolder(train_dir, transform=transform)

    # Split dataset into training and validation
    train_size = int((1 - val_split) * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # Initialize model, loss function, and optimizer
    model = VGG16(num_classes=len(dataset.classes)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=initial_lr, momentum=0.9)
    scaler = amp.GradScaler()

    start_time = time.time()

    # Training loop
    for epoch in range(1, epochs + 1):
        if epoch % 5 == 0:  # Adjust learning rate every 5 epochs
            for param_group in optimizer.param_groups:
                param_group['lr'] /= 1.5

        model.train()
        running_loss, running_error, num_batches = 0, 0, 0

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}")
        for minibatch_data, minibatch_label in progress_bar:
            optimizer.zero_grad()

            minibatch_data = minibatch_data.to(device, non_blocking=True)
            minibatch_label = minibatch_label.to(device, non_blocking=True)

            with amp.autocast():
                outputs = model(minibatch_data)
                loss = criterion(outputs, minibatch_label)

            # Backpropagation
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # Compute loss and error
            running_loss += loss.detach().item()
            _, predicted = torch.max(outputs.detach(), 1)
            running_error += (predicted != minibatch_label).sum().item()
            num_batches += 1

            # Update progress bar
            progress_bar.set_postfix(loss=running_loss / num_batches, error=running_error / len(train_dataset) * 100)

        # Compute epoch statistics
        total_loss = running_loss / num_batches
        total_error = running_error / len(train_dataset)
        elapsed_time = (time.time() - start_time) / 60

        print(f"Epoch={epoch}, Time={elapsed_time:.2f} min, LR={optimizer.param_groups[0]['lr']:.6f}, Loss={total_loss:.4f}, Error={total_error * 100:.2f}%")

        # Evaluate model on validation set
        evaluate_model(model, val_loader, device, mode='Validation')

    return model

In [ ]:
batch_size = 100
epochs = 10
initial_lr = 0.03
Vgg16Model = train_model(train_dir, batch_size, epochs, initial_lr, device=device)

<ipython-input-18-e19f2c363ab3>:43: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()
Epoch 1:   0%|          | 0/247 [00:00<?, ?it/s]<ipython-input-18-e19f2c363ab3>:63: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
Epoch 1: 100%|██████████| 247/247 [03:14<00:00,  1.27it/s, error=96.9, loss=4.25]


Epoch=1, Time=3.24 min, LR=0.030000, Loss=4.2520, Error=96.90%


Evaluating Validation: 100%|██████████| 62/62 [00:38<00:00,  1.61it/s]


Validation Accuracy: 8.89%


Epoch 2: 100%|██████████| 247/247 [03:15<00:00,  1.27it/s, error=80.3, loss=3.25]


...

Validation Accuracy: 62.82%


Epoch 10: 100%|██████████| 247/247 [03:12<00:00,  1.28it/s, error=6.3, loss=0.204] 


Epoch=10, Time=38.16 min, LR=0.013333, Loss=0.2043, Error=6.30%


Evaluating Validation: 100%|██████████| 62/62 [00:39<00:00,  1.59it/s]

Validation Accuracy: 66.11%


In [ ]:
# Evaluate the model
evaluate_model(Vgg16Model, test_loader, device)

Evaluating Test: 100%|██████████| 78/78 [00:47<00:00,  1.65it/s]

Test Accuracy: 50.10%


50.10384215991693

In [ ]:
save_model(Vgg16Model, 'vgg16_model.pth')

Model saved to vgg16_model.pth
